# Д/З №5. Работа с "Черным ящиком"
### ( ◔ ⌣ ◔ )

В этой (очень экспериментальной!) домашке вам предстоит работать с "черным ящиком".

In [1]:
import requests
import numpy as np
import pandas as pd

### Датасет и целевая переменная

Данные имеют следующую структуру:

Признаки:
1. `'scint_alpha'`
2. `'drift_bias'`
3. `'pixel_jitter'`
4. `'muon_flux'`
5. `'cathode_temp'`
6. `'anode_noise'`
7. `'beam_phase'`
8. `'shield_rho'`
9. `'veto_count'`
10. `'gain_offset'`
11. `'cerenkov_angle'`
12. `'trigger_rate'`
13. `'track_curvature'`

Целевая переменная: `'y'`.

Но где же датасет? Датасета у вас не будет... В этой задаче вам надо его составить самостоятельно.

### Контест

Вам предоставляется API и код для запроса значений `y` у сервера по предложенным вами произвольным значениям `x`.

У вас будет 10 дней (возможно, немного больше...) на то, чтобы придумать, что надо запросить для обучения модели.

После этого удаленный сервер будет **закрыт**, и будет открыт контест на Kaggle, в котором будут даны точки тестовой выборки. Какие это будут точки, не раскрывается (иначе их можно было бы запросить в период тренировки).

Функцией ошибки на Kaggle будет MSE.

### Работа с API

Правила работы:

* Можно пользоваться только своим аккаунтом
* Заданы лимиты: не более 5 тыс. точек **и** не более 50 запросов в сутки.
* В одном запросе не более 1000 точек (если запрашиваете больше за раз, будет ошибка, и потратится квота, следите за лимитами).
* Если вы потеряли результаты запросов или сделали бессмысленные запросы, лимиты не компенсируются.
    * Следствие: не теряйте результаты запросов.
* Лимиты сбрасываются в полночь по UTC.

### Авторизация и аутентификация

В нормальной ситуации слать свой пароль ни проверяющему, ни кому-то другому не надо, но в этой ситуации прошу вас это сделать -- сервер к моменту проверки будет закрыт, а наличие пароля в коде облегчит проверку.

In [2]:
API_URL  = 'https://aim.bioml.ru'
USERNAME = 'vlad.lazar.ik529'
PASSWORD = 'timcD84nvyYC'

FEATURE_NAMES = [
    'scint_alpha', 'drift_bias', 'pixel_jitter', 'muon_flux',
    'cathode_temp', 'anode_noise', 'beam_phase', 'shield_rho',
    'veto_count', 'gain_offset', 'cerenkov_angle', 'trigger_rate',
    'track_curvature',
]

resp = requests.post(
    f'{API_URL}/token',
    data={'username': USERNAME, 'password': PASSWORD},
)
resp.raise_for_status()

TOKEN   = resp.json()['access_token']
HEADERS = {'Authorization': f'Bearer {TOKEN}'}
print('Authenticated. Token starts with:', TOKEN[:3], '...')

Authenticated. Token starts with: eyJ ...


### Проверка квоты

In [3]:
quota = requests.get(f'{API_URL}/quota', headers=HEADERS).json()

print(f"Requests : {quota['requests_made_today']} used / {quota['daily_request_limit']} limit  "
      f"({quota['requests_remaining']} remaining)")
print(f"Points   : {quota['points_used_today']} used / {quota['daily_point_limit']} limit  "
      f"({quota['points_remaining']} remaining)")

Requests : 3 used / 50 limit  (47 remaining)
Points   : 3000 used / 5000 limit  (2000 remaining)


### Запрос на сервер

_Внимание!_ Вызовы функции, приведенной ниже, будут тратить вашу квоту!

In [4]:
# Helper function: query a batch, return (X_ok, y_ok) with null rows dropped
def query(X: np.ndarray):
    """X: shape (N, 13). Returns (X_valid, y_valid, quota_dict)."""
    assert X.ndim == 2 and X.shape[1] == 13
    assert len(X) <= 1000

    resp = requests.post(
        f'{API_URL}/query',
        headers=HEADERS,
        json={'points': X.tolist()},
    )
    resp.raise_for_status()
    data   = resp.json()
    values = data['values']
    quota  = data['quota']

    mask = [v is not None for v in values]
    X_ok = X[mask]
    y_ok = np.array([v for v in values if v is not None])

    n_null = quota['nulls_in_this_response']
    if n_null:
        print(f'Warning: {n_null} point(s) returned null - daily quota reached.')
    print(f'Requests remaining today: {quota["requests_remaining"]}')
    print(f'Points remaining today:   {quota["points_remaining"]}')

    return X_ok, y_ok

In [5]:
# Query 10 random points from N(0, 1)
# rng  = np.random.default_rng(seed=42)
# X_10 = rng.standard_normal((10, 13))

# X_ok, y_ok = query(X_10)
# result = pd.DataFrame(X_ok, columns=FEATURE_NAMES).assign(y=y_ok)

# print(f'Queried {len(X_10)} points, received {len(y_ok)} values.')
# print()
# result.head()

### История запросов

In [6]:
history = requests.get(
    f'{API_URL}/history',
    headers=HEADERS,
    params={'limit': 50},
).json()

df_hist = pd.DataFrame(history)
df_hist['ts'] = pd.to_datetime(df_hist['ts'])
print(f'{len(df_hist)} requests in history (most recent 50 shown)')
df_hist

20 requests in history (most recent 50 shown)


,id,ts,n_points_real,n_points_null
0,680,2026-05-02 14:14:12.939036+00:00,1000,0
1,679,2026-05-02 14:12:44.524182+00:00,1000,0
2,677,2026-05-02 14:06:52.911454+00:00,1000,0
3,308,2026-05-01 00:11:19.640375+00:00,999,0
4,307,2026-05-01 00:11:04.058647+00:00,998,0
5,306,2026-05-01 00:06:55.898376+00:00,998,0
6,305,2026-05-01 00:02:56.762573+00:00,200,0
7,304,2026-05-01 00:02:26.450552+00:00,200,0
8,303,2026-05-01 00:02:05.341195+00:00,200,0
9,302,2026-05-01 00:01:45.283069+00:00,200,0


In [7]:
import os
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import qmc

from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, root_mean_squared_error
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler


DATA_DIR = Path("active_learning_data")
DATA_DIR.mkdir(exist_ok=True)

REQUESTED_PATH = DATA_DIR / "requested.csv"
VALID_PATH = DATA_DIR / "valid.csv"
META_PATH = DATA_DIR / "meta.json"


def load_requested() -> pd.DataFrame:
    if REQUESTED_PATH.exists():
        return pd.read_csv(REQUESTED_PATH)
    return pd.DataFrame(columns=FEATURE_NAMES)


def load_valid() -> pd.DataFrame:
    if VALID_PATH.exists():
        return pd.read_csv(VALID_PATH)
    return pd.DataFrame(columns=FEATURE_NAMES + ["y"])


def save_requested_append(X: np.ndarray):
    df = pd.DataFrame(X, columns=FEATURE_NAMES)
    header = not REQUESTED_PATH.exists()
    df.to_csv(REQUESTED_PATH, mode="a", index=False, header=header)


def save_valid_append(X_ok: np.ndarray, y_ok: np.ndarray):
    df = pd.DataFrame(X_ok, columns=FEATURE_NAMES)
    df["y"] = y_ok
    header = not VALID_PATH.exists()
    df.to_csv(VALID_PATH, mode="a", index=False, header=header)


def safe_query_and_save(X: np.ndarray):
    """
    Обёртка над query из ноутбука.
    Всегда сохраняет:
    - все отправленные X;
    - только валидные X_ok, y_ok.
    """
    X = np.asarray(X, dtype=float)
    assert X.ndim == 2
    assert X.shape[1] == len(FEATURE_NAMES)
    assert len(X) <= 1000

    save_requested_append(X)

    X_ok, y_ok = query(X)

    save_valid_append(X_ok, y_ok)

    print(f"Requested: {len(X)}")
    print(f"Received valid: {len(X_ok)}")
    print(f"Invalid/null: {len(X) - len(X_ok)}")
    print(f"Valid ratio: {len(X_ok) / max(len(X), 1):.4f}")

    return X_ok, y_ok

In [8]:
def sobol_normal_pool(n: int, d: int, scale: float = 1.0, seed: int = 42) -> np.ndarray:
    """
    Генерирует quasi-random точки примерно из N(0, scale^2).
    Через inverse CDF нормального распределения.
    """
    from scipy.special import ndtri

    sampler = qmc.Sobol(d=d, scramble=True, seed=seed)

    m = int(np.ceil(np.log2(n)))
    u = sampler.random_base2(m=m)[:n]

    eps = 1e-9
    u = np.clip(u, eps, 1.0 - eps)

    X = ndtri(u) * scale
    return X


def sobol_uniform_box(n: int, low: np.ndarray, high: np.ndarray, seed: int = 42) -> np.ndarray:
    """
    Равномерное quasi-random покрытие прямоугольной области.
    """
    low = np.asarray(low, dtype=float)
    high = np.asarray(high, dtype=float)

    d = len(low)
    sampler = qmc.Sobol(d=d, scramble=True, seed=seed)

    m = int(np.ceil(np.log2(n)))
    u = sampler.random_base2(m=m)[:n]

    return low + u * (high - low)


def make_initial_batches():
    """
    Разумный первый день:
    - 500 точек из N(0, 1)
    - 500 точек из N(0, 2)
    - 1000 точек из N(0, 0.5)
    
    Не отправляй всё вслепую, если первый батч дал много invalid.
    """
    d = len(FEATURE_NAMES)

    batches = [
        sobol_normal_pool(500, d, scale=1.0, seed=101),
        sobol_normal_pool(500, d, scale=2.0, seed=102),
        sobol_normal_pool(1000, d, scale=0.5, seed=103),
    ]

    return batches

In [9]:
# initial_batches = make_initial_batches()

# X0 = initial_batches[0]
# X_ok, y_ok = safe_query_and_save(X0)

In [10]:
# X1 = initial_batches[1]
# X_ok, y_ok = safe_query_and_save(X1)

In [11]:
import numpy as np
import pandas as pd
from scipy.stats import qmc
from scipy.special import ndtri


def _sobol_uniform(n: int, d: int, seed: int = 42) -> np.ndarray:
    sampler = qmc.Sobol(d=d, scramble=True, seed=seed)
    m = int(np.ceil(np.log2(max(n, 2))))
    return sampler.random_base2(m=m)[:n]


def _sobol_normal(n: int, d: int, seed: int = 42) -> np.ndarray:
    u = _sobol_uniform(n=n, d=d, seed=seed)
    u = np.clip(u, 1e-9, 1.0 - 1e-9)
    return ndtri(u)


def _drop_already_requested(X: np.ndarray, decimals: int = 12) -> np.ndarray:
    """
    Убирает точки, которые уже были отправлены раньше.
    Работает через округление, чтобы избежать проблем float-сравнения.
    """
    X = np.asarray(X, dtype=float)

    try:
        df_req = load_requested()
    except Exception:
        df_req = pd.DataFrame(columns=FEATURE_NAMES)

    if len(df_req) == 0:
        return X

    req_arr = df_req[FEATURE_NAMES].to_numpy(dtype=float)

    requested_keys = set(map(tuple, np.round(req_arr, decimals=decimals)))
    new_keys = list(map(tuple, np.round(X, decimals=decimals)))

    mask = np.array([key not in requested_keys for key in new_keys], dtype=bool)
    return X[mask]


def _deduplicate_rows(X: np.ndarray, decimals: int = 12) -> np.ndarray:
    df = pd.DataFrame(np.asarray(X, dtype=float), columns=FEATURE_NAMES)
    df = df.round(decimals).drop_duplicates()
    return df[FEATURE_NAMES].to_numpy(dtype=float)


def make_controlled_expansion_batch(
    batch_size: int = 1000,
    seed: int = 2026,
    box_expand: float = 1.50,
    radial_radii=(2.5, 3.5, 5.0),
    wide_normal_scale: float = 3.0,
    core_noise_scale: float = 0.35,
):
    """
    Создаёт следующий батч для расширения области.

    Состав:
    40% — expanded box вокруг уже валидных точек
    35% — radial rays наружу от центра
    15% — wide normal cloud
    10% — local safe core

    Требует:
    - FEATURE_NAMES
    - load_valid()
    - load_requested()
    """

    rng = np.random.default_rng(seed)

    df_valid = load_valid()

    if len(df_valid) == 0:
        raise RuntimeError("No valid data found. First collect at least one valid batch.")

    X_valid = df_valid[FEATURE_NAMES].to_numpy(dtype=float)
    d = len(FEATURE_NAMES)

    center = np.median(X_valid, axis=0)

    q05 = np.quantile(X_valid, 0.05, axis=0)
    q95 = np.quantile(X_valid, 0.95, axis=0)

    half_width = 0.5 * (q95 - q05)
    std = np.std(X_valid, axis=0)

    # Защита от нулевых масштабов
    global_scale = np.nanmedian(std[std > 1e-12]) if np.any(std > 1e-12) else 1.0
    std = np.where(std > 1e-12, std, global_scale)
    half_width = np.where(half_width > 1e-12, half_width, std)

    n_box = int(batch_size * 0.40)
    n_radial = int(batch_size * 0.35)
    n_wide = int(batch_size * 0.15)
    n_core = batch_size - n_box - n_radial - n_wide

    parts = []

    # 1. Expanded box
    # Не строим box по min/max, потому что выбросы могут сильно раздувать область.
    low = center - box_expand * half_width
    high = center + box_expand * half_width

    u_box = _sobol_uniform(n_box, d, seed=seed + 1)
    X_box = low + u_box * (high - low)
    parts.append(X_box)

    # 2. Radial rays
    # Идём от центра наружу по случайным направлениям.
    directions = rng.normal(size=(n_radial, d))
    directions_norm = np.linalg.norm(directions, axis=1, keepdims=True)
    directions_norm = np.where(directions_norm > 1e-12, directions_norm, 1.0)
    directions = directions / directions_norm

    radii = np.array(radial_radii, dtype=float)
    chosen_radii = rng.choice(radii, size=n_radial, replace=True)

    # Масштабируем направление по std каждого признака.
    X_radial = center + directions * chosen_radii[:, None] * std[None, :]
    parts.append(X_radial)

    # 3. Wide normal cloud
    # Более широкое облако вокруг центра.
    Z_wide = _sobol_normal(n_wide, d, seed=seed + 2)
    X_wide = center + wide_normal_scale * Z_wide * std
    parts.append(X_wide)

    # 4. Safe core
    # Немного точек рядом с уже валидными точками, чтобы не потратить весь батч на границу.
    anchor_idx = rng.choice(len(X_valid), size=n_core, replace=True)
    anchors = X_valid[anchor_idx]

    noise = rng.normal(
        loc=0.0,
        scale=core_noise_scale,
        size=(n_core, d),
    ) * std

    X_core = anchors + noise
    parts.append(X_core)

    X = np.vstack(parts)

    X = _deduplicate_rows(X)
    X = _drop_already_requested(X)

    # Если после удаления дублей стало меньше batch_size, добиваем radial-точками.
    attempt = 0
    while len(X) < batch_size and attempt < 10:
        missing = batch_size - len(X)

        extra_dirs = rng.normal(size=(missing * 2, d))
        extra_dirs_norm = np.linalg.norm(extra_dirs, axis=1, keepdims=True)
        extra_dirs_norm = np.where(extra_dirs_norm > 1e-12, extra_dirs_norm, 1.0)
        extra_dirs = extra_dirs / extra_dirs_norm

        extra_radii = rng.choice(np.array(radial_radii, dtype=float) * (1.0 + 0.25 * attempt), size=missing * 2, replace=True)
        X_extra = center + extra_dirs * extra_radii[:, None] * std[None, :]

        X = np.vstack([X, X_extra])
        X = _deduplicate_rows(X)
        X = _drop_already_requested(X)

        attempt += 1

    if len(X) > batch_size:
        X = X[:batch_size]

    df_batch = pd.DataFrame(X, columns=FEATURE_NAMES)

    print("Created expansion batch")
    print("Batch size:", len(df_batch))
    print("box_expand:", box_expand)
    print("radial_radii:", radial_radii)
    print("wide_normal_scale:", wide_normal_scale)
    print("core_noise_scale:", core_noise_scale)

    print("\nBatch ranges:")
    display(pd.DataFrame({
        "feature": FEATURE_NAMES,
        "batch_min": df_batch.min(axis=0).to_numpy(),
        "batch_median": df_batch.median(axis=0).to_numpy(),
        "batch_max": df_batch.max(axis=0).to_numpy(),
    }))

    return df_batch[FEATURE_NAMES].to_numpy(dtype=float)

In [12]:
# X_expand = make_controlled_expansion_batch(
#     batch_size=1000,
#     seed=3001,
#     box_expand=1.50,
#     radial_radii=(2.5, 3.5, 5.0),
#     wide_normal_scale=3.0,
#     core_noise_scale=0.35,
# )

# X_ok, y_ok = safe_query_and_save(X_expand)

In [13]:
# X_expand = make_controlled_expansion_batch(
#     batch_size=200,
#     seed=3002,
#     box_expand=2.25,
#     radial_radii=(4.0, 6.0, 8.0),
#     wide_normal_scale=5.0,
#     core_noise_scale=0.50,
# )

# X_ok, y_ok = safe_query_and_save(X_expand)

In [14]:
# X_expand = make_controlled_expansion_batch(
#     batch_size=1000,
#     seed=3002,
#     box_expand=22.5,
#     radial_radii=(8.0, 12.0, 16.0),
#     wide_normal_scale=10.0,
#     core_noise_scale=1.0,
# )

# X_ok, y_ok = safe_query_and_save(X_expand)

In [15]:
# X_expand = make_controlled_expansion_batch(
#     batch_size=200,
#     seed=3003,
#     box_expand=45.0,
#     radial_radii=(10.0, 15.0, 20.0),
#     wide_normal_scale=15.0,
#     core_noise_scale=1.5,
# )

# X_ok, y_ok = safe_query_and_save(X_expand)

In [16]:
# X_expand = make_controlled_expansion_batch(
#     batch_size=200,
#     seed=3004,
#     box_expand=50.0,
#     radial_radii=(100.0, 150.0, 200.0),
#     wide_normal_scale=150.0,
#     core_noise_scale=15,
# )

# X_ok, y_ok = safe_query_and_save(X_expand)

In [17]:
# X_expand = make_controlled_expansion_batch(
#     batch_size=200,
#     seed=3005,
#     box_expand=500.0,
#     radial_radii=(1000.0, 1500.0, 2000.0),
#     wide_normal_scale=1500.0,
#     core_noise_scale=150,
# )

# X_ok, y_ok = safe_query_and_save(X_expand)

In [18]:
# X_expand = make_controlled_expansion_batch(
#     batch_size=200,
#     seed=3006,
#     box_expand=5000.0,
#     radial_radii=(10000.0, 15000.0, 20000.0),
#     wide_normal_scale=15000.0,
#     core_noise_scale=1500,
# )

# X_ok, y_ok = safe_query_and_save(X_expand)

In [19]:
# X_expand = make_controlled_expansion_batch(
#     batch_size=200,
#     seed=3007,
#     box_expand=500000.0,
#     radial_radii=(1000000.0, 1500000.0, 2000000.0),
#     wide_normal_scale=1500000.0,
#     core_noise_scale=150000,
# )

# X_ok, y_ok = safe_query_and_save(X_expand)

In [20]:
# X_expand = make_controlled_expansion_batch(
#     batch_size=200,
#     seed=3008,
#     box_expand=5000000.0,
#     radial_radii=(10000000.0, 15000000.0, 20000000.0),
#     wide_normal_scale=15000000.0,
#     core_noise_scale=1500000,
# )

# X_ok, y_ok = safe_query_and_save(X_expand)

# Основная грязь

In [21]:
def row_key_df(df: pd.DataFrame, decimals: int = 12):
    arr = df[FEATURE_NAMES].to_numpy(dtype=float)
    arr = np.round(arr, decimals=decimals)
    return list(map(tuple, arr))


def make_validity_dataset():
    df_req = load_requested()
    df_val = load_valid()

    if len(df_req) == 0:
        return None, None

    valid_keys = set(row_key_df(df_val))
    req_keys = row_key_df(df_req)

    y_validity = np.array([1 if k in valid_keys else 0 for k in req_keys], dtype=int)
    X_validity = df_req[FEATURE_NAMES].to_numpy(dtype=float)

    return X_validity, y_validity


def fit_validity_model():
    Xv, yv = make_validity_dataset()

    if Xv is None or len(Xv) < 50:
        print("Not enough requested points for validity model.")
        return None

    if len(np.unique(yv)) < 2:
        print("Only one validity class observed. Validity model is not needed yet.")
        return None

    model = ExtraTreesClassifier(
        n_estimators=800,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        # n_jobs=-1,
    )
    model.fit(Xv, yv)

    print("Validity model fitted.")
    print("Requested points:", len(Xv))
    print("Valid share:", yv.mean())

    return model

In [22]:
def fit_regression_ensemble():
    df = load_valid()

    if len(df) < 100:
        print("Not enough valid points for regression ensemble.")
        return None

    X = df[FEATURE_NAMES].to_numpy(dtype=float)
    y = df["y"].to_numpy(dtype=float)

    models = []

    for seed in [11, 22, 33, 44, 55]:
        model = ExtraTreesRegressor(
            n_estimators=700,
            min_samples_leaf=2,
            max_features=0.8,
            bootstrap=True,
            random_state=seed,
            n_jobs=-1,
        )
        model.fit(X, y)
        models.append(model)

    print(f"Regression ensemble fitted on {len(X)} valid points.")

    if len(X) >= 300:
        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.25,
            random_state=42,
        )

        check_model = ExtraTreesRegressor(
            n_estimators=700,
            min_samples_leaf=2,
            max_features=0.8,
            bootstrap=True,
            random_state=777,
            n_jobs=-1,
        )
        check_model.fit(X_train, y_train)
        pred = check_model.predict(X_test)
        rmse = root_mean_squared_error(y_test, pred)
        mse = mean_squared_error(y_test, pred)

        print(f"Local holdout MSE : {mse:.6f}")
        print(f"Local holdout RMSE: {rmse:.6f}")

    return models


def ensemble_predict_mean_std(models, X: np.ndarray):
    preds = np.column_stack([m.predict(X) for m in models])
    return preds.mean(axis=1), preds.std(axis=1)

In [23]:
def estimate_current_box(expand: float = 0.20):
    """
    Строит текущую прямоугольную область по уже валидным точкам.
    Потом немного расширяет её.
    """
    df = load_valid()

    if len(df) < 20:
        d = len(FEATURE_NAMES)
        return np.full(d, -2.0), np.full(d, 2.0)

    X = df[FEATURE_NAMES].to_numpy(dtype=float)

    low = np.quantile(X, 0.01, axis=0)
    high = np.quantile(X, 0.99, axis=0)

    width = high - low
    width = np.where(width <= 1e-9, 1.0, width)

    low = low - expand * width
    high = high + expand * width

    return low, high


def make_candidate_pool(
    n_pool: int = 100_000,
    seed: int = 42,
    normal_fraction: float = 0.35,
):
    """
    Кандидаты берутся из смеси:
    - текущая найденная box-область;
    - нормальное облако вокруг уже валидных точек.
    """
    d = len(FEATURE_NAMES)
    low, high = estimate_current_box(expand=0.25)

    n_normal = int(n_pool * normal_fraction)
    n_box = n_pool - n_normal

    X_box = sobol_uniform_box(n_box, low=low, high=high, seed=seed)

    df = load_valid()
    if len(df) >= 20:
        X_valid = df[FEATURE_NAMES].to_numpy(dtype=float)

        center = X_valid[np.random.default_rng(seed).choice(len(X_valid), size=n_normal, replace=True)]

        std = np.std(X_valid, axis=0)
        std = np.where(std <= 1e-9, 1.0, std)

        noise = np.random.default_rng(seed + 1).normal(
            loc=0.0,
            scale=0.25 * std,
            size=(n_normal, d),
        )

        X_normal = center + noise
    else:
        X_normal = sobol_normal_pool(n_normal, d=d, scale=1.0, seed=seed + 2)

    X_pool = np.vstack([X_box, X_normal])
    return X_pool


def select_diverse_top_points(X_pool, score, batch_size: int, top_multiplier: int = 20):
    """
    Берём top по score, затем диверсифицируем через MiniBatchKMeans.
    """
    n = len(X_pool)
    batch_size = min(batch_size, n)

    if n <= batch_size:
        return X_pool

    top_n = min(max(batch_size * top_multiplier, 5000), n)
    top_idx = np.argsort(score)[-top_n:]

    X_top = X_pool[top_idx]
    score_top = score[top_idx]

    scaler = StandardScaler()
    X_top_scaled = scaler.fit_transform(X_top)

    kmeans = MiniBatchKMeans(
        n_clusters=batch_size,
        random_state=42,
        batch_size=4096,
        n_init="auto",
    )
    labels = kmeans.fit_predict(X_top_scaled)

    selected = []

    for k in range(batch_size):
        local_idx = np.where(labels == k)[0]
        if len(local_idx) == 0:
            continue

        best_local = local_idx[np.argmax(score_top[local_idx])]
        selected.append(best_local)

    selected = np.array(selected, dtype=int)

    if len(selected) < batch_size:
        missing = batch_size - len(selected)
        rest = np.setdiff1d(np.arange(len(X_top)), selected)
        extra = rest[np.argsort(score_top[rest])[-missing:]]
        selected = np.concatenate([selected, extra])

    return X_top[selected]


def make_active_batch(
    batch_size: int = 1000,
    n_pool: int = 100_000,
    min_valid_proba: float = 0.70,
    boundary_fraction: float = 0.15,
    random_fraction: float = 0.10,
    seed: int = 42,
):
    """
    Итоговый batch:
    - основная часть: uncertainty * P(valid)
    - часть около границы валидности
    - часть random exploration
    """
    assert 0.0 <= boundary_fraction <= 1.0
    assert 0.0 <= random_fraction <= 1.0
    assert boundary_fraction + random_fraction < 1.0

    validity_model = fit_validity_model()
    reg_models = fit_regression_ensemble()

    X_pool = make_candidate_pool(n_pool=n_pool, seed=seed)

    if validity_model is not None:
        p_valid = validity_model.predict_proba(X_pool)[:, list(validity_model.classes_).index(1)]
    else:
        p_valid = np.ones(len(X_pool), dtype=float)

    if reg_models is not None:
        _, uncertainty = ensemble_predict_mean_std(reg_models, X_pool)
    else:
        uncertainty = np.ones(len(X_pool), dtype=float)

    n_random = int(batch_size * random_fraction)
    n_boundary = int(batch_size * boundary_fraction)
    n_active = batch_size - n_random - n_boundary

    rng = np.random.default_rng(seed)

    selected_parts = []

    # 1. Active uncertainty part
    active_mask = p_valid >= min_valid_proba

    if active_mask.sum() < n_active:
        active_mask = p_valid >= np.quantile(p_valid, 0.50)

    X_active_pool = X_pool[active_mask]
    p_active = p_valid[active_mask]
    u_active = uncertainty[active_mask]

    active_score = u_active * np.clip(p_active, 0.05, 1.0)

    X_active = select_diverse_top_points(
        X_active_pool,
        active_score,
        batch_size=n_active,
        top_multiplier=25,
    )
    selected_parts.append(X_active)

    # 2. Boundary validity part: p_valid около 0.4–0.8
    if n_boundary > 0:
        boundary_score = -np.abs(p_valid - 0.60)
        boundary_mask = p_valid >= 0.25

        if boundary_mask.sum() >= n_boundary:
            X_boundary_pool = X_pool[boundary_mask]
            score_boundary = boundary_score[boundary_mask]
        else:
            X_boundary_pool = X_pool
            score_boundary = boundary_score

        X_boundary = select_diverse_top_points(
            X_boundary_pool,
            score_boundary,
            batch_size=n_boundary,
            top_multiplier=20,
        )
        selected_parts.append(X_boundary)

    # 3. Random exploration part, but not completely suicidal
    if n_random > 0:
        random_mask = p_valid >= max(0.30, min_valid_proba - 0.30)

        if random_mask.sum() < n_random:
            random_mask = np.ones(len(X_pool), dtype=bool)

        X_random_pool = X_pool[random_mask]
        idx = rng.choice(len(X_random_pool), size=min(n_random, len(X_random_pool)), replace=False)
        X_random = X_random_pool[idx]
        selected_parts.append(X_random)

    X_batch = np.vstack(selected_parts)

    # Удалим полные дубликаты после округления
    X_batch_df = pd.DataFrame(X_batch, columns=FEATURE_NAMES)
    X_batch_df = X_batch_df.round(12).drop_duplicates()

    if len(X_batch_df) > batch_size:
        X_batch_df = X_batch_df.iloc[:batch_size]

    print("Batch created:")
    print("size:", len(X_batch_df))
    print("estimated mean P(valid):", float(np.mean(p_valid)))
    print("estimated pool P(valid) quantiles:", np.quantile(p_valid, [0.1, 0.5, 0.9]))

    return X_batch_df[FEATURE_NAMES].to_numpy(dtype=float)

In [24]:
# X_next = make_active_batch(
#     batch_size=1000,
#     n_pool=100_000,
#     min_valid_proba=0.70,
#     boundary_fraction=0.15,
#     random_fraction=0.10,
#     seed=2026,
# )

# X_ok, y_ok = safe_query_and_save(X_next)

In [25]:
# X_next = make_active_batch(
#     batch_size=1000,
#     n_pool=100_000,
#     min_valid_proba=0.70,
#     boundary_fraction=0.15,
#     random_fraction=0.10,
#     seed=2026,
# )

# X_ok, y_ok = safe_query_and_save(X_next)

In [26]:
# X_next = make_active_batch(
#     batch_size=1000,
#     n_pool=100_000,
#     min_valid_proba=0.70,
#     boundary_fraction=0.15,
#     random_fraction=0.10,
#     seed=2026,
# )

# X_ok, y_ok = safe_query_and_save(X_next)

In [ ]:
from skactiveml.pool import GreedySamplingX


def make_greedy_sampling_x_batch(
    batch_size: int = 1000,
    n_pool: int = 50_000,
    min_valid_proba: float = 0.70,
    seed: int = 42,
):
    validity_model = fit_validity_model()

    X_pool = make_candidate_pool(n_pool=n_pool, seed=seed)

    if validity_model is not None:
        p_valid = validity_model.predict_proba(X_pool)[:, list(validity_model.classes_).index(1)]
        X_pool = X_pool[p_valid >= min_valid_proba]

    if len(X_pool) <= batch_size:
        return X_pool

    df_valid = load_valid()

    if len(df_valid) > 0:
        X_labeled = df_valid[FEATURE_NAMES].to_numpy(dtype=float)
        y_labeled = df_valid["y"].to_numpy(dtype=float)

        X_all = np.vstack([X_labeled, X_pool])
        y_all = np.concatenate([y_labeled, np.full(len(X_pool), np.nan)])

        candidate_indices = np.arange(len(X_labeled), len(X_all))
    else:
        X_all = X_pool
        y_all = np.full(len(X_pool), np.nan)
        candidate_indices = np.arange(len(X_all))

    qs = GreedySamplingX(random_state=seed)

    query_idx = qs.query(
        X=X_all,
        y=y_all,
        candidates=candidate_indices,
        batch_size=batch_size,
    )

    return X_all[query_idx]

In [ ]:
X_next = make_greedy_sampling_x_batch(
    batch_size=1000,
    n_pool=50_000,
    min_valid_proba=0.60,
    seed=7001,
)

X_ok, y_ok = safe_query_and_save(X_next)

Validity model fitted.
Requested points: 10395
Valid share: 0.801058201058201
HERE!!!!!!!!!!!!!!!!!!!
1
HERE!!!!!!!!!!!!!!!!!!!
